In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow, imshow_multiple, InfinitePhaseScreen
from AI4AO.paths import DATA_DIR
from AI4AO.Utils import seed_everything
from AI4AO.LossFunctions import LogResidualVarianceLoss

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import numpy as np

# %% Set general parameter


## WFS and Telescope parameters
WFSParams = dict(
    {
        "Nres":100,
        "sampling":140/40,
        "D": 1.82,
        "centralObstruction": 0.3,
        "useNoise": True,
        "Modulation": 0,
        "Wavelength": 635e-9,
        "Nphotons": [4.5, 6],  # Log range of number of photons in measurement
        "RON": [1, 3],  # Read-out noise in photons per pixel per frame
        "Substract_Reference": True,  # Substract or not the reference intensity frame
        "Extract_pupils": True,
        "Extract_pupils_pad": 6,
        "Center_noise": 2,
        "Pupil_size_noise": 0.05,  # +/-5% pupil size jitter, e.g. 38-42 px for a 40 px pupil
        "Bin_factor": 1
    }
)


## Atmosphere parameters
AtmosParams = dict(
    {
        "r0": [0.05, 0.05],  # Fired parameter range (m)
        "L0": [20, 30.0000],  # Outter scale range (m)
        "Nphases": 1,  # Number of phases in the batch
        "Layers": [5, 6],  # Number of layers in phase range
        "f_slope": 11.0 / 6.0,  # Slope of the spectrum (11/6) is the default
        "Scintillation": False # Use or not scinitillation
    }
)


## Loop parameters
LoopParams = dict(
    {
        "loopFrequency": 500,
        "delayFrames": 1,
        "windSpeedVector": [1, 10],
        "levelOfCorrection": [0.0, 1.0],
        "loopGain": [0.2, 0.5],
        "loopLeak": [0.9,1.] 
    }
)

## Atmosphere parameters
DMParams = dict(
    {
        "Nactuator": 24,
        "Nmodes": 400,
        "moffatParam": 2,
        "signedAmplitude": -5e-6,
        "MechCoupling": 0.36,
        "FlipLeftRight": False,
        "FlipTopBottom": False
    }
)

TrainParams = dict(
    {
        "lro": 2e-4,
        "lrn": 4e-4,
        "TrainRunNb": 5000,
        "TestRunNb": 10,
        "OptimizeMask": False,
    }
)


In [ ]:
# dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
# dataset.generateClosedLoop = True

dataset = InfinitePhaseScreen(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.subharmonicLevels=3

In [ ]:
PATH = DATA_DIR / "Ekarus"

# WFS creation
wfs = PyramidWFS(WFSParams, device)
wfs.LoadCalibration(PATH / "EkarusWFS.pth")
wfs.eval()
wfs.modulation = 5

dm = DeformableMirror(WFSParams, DMParams, device)
dm.LoadCalibration(PATH / "EkarusDM.pth")
dm.eval()

M2C_Zernike = dm.MakeZernikeM2C(200)

In [ ]:
# M2C = np.load(PATH / "M2C_KL_OOPAO.npy")
# M2C = torch.from_numpy(M2C).to(device=device, dtype=torch.float32)
# M2C = M2C[:, :DMParams["Nmodes"]]  # keep only the modes this reconstructor is trained to output

M2C = M2C_Zernike

z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))  # full-resolution phase -> modal coefficients

In [ ]:
nModes = 200
modes = dm(M2C.T[:nModes])
wfs.BuildReconstructionMatrix(modes)

In [ ]:
M2C_T = M2C.T[:nModes]

coefs = torch.zeros((dataset.Nphases, nModes), device=device)

gain = 0.3
leak = 0.995

n_iter = 1000

seed_everything(seed=2026)

defocus = modes[2] * 30 # Modes are normalized to have unitary variance
astigmatism = modes[3] * 1

batch = dataset[0]
residual_opd = torch.clone(batch["opd"])

psf_frame = wfs.GetPSF(residual_opd)*0
psf_frame_defocus_positive = wfs.GetPSF(residual_opd + defocus)*0
psf_frame_defocus_negative = wfs.GetPSF(residual_opd - defocus)*0

zernike_telemetry = []

for i in tqdm(range(n_iter)):
    batch = dataset[i]
    opd = batch["opd"]
    # wfs_frame = wfs(residual_opd)
    # reconstruction = wfs_frame.flatten(start_dim=-2) @ wfs.reconstructionMatrix
    reconstruction = residual_opd.flatten(start_dim=-2) @ z_inv[:, :nModes]   # Perfect recosntruction

    coefs = leak * coefs + gain * reconstruction
    
    reconstructed_opd = dm(coefs @ M2C_T)
    residual_opd = opd - reconstructed_opd

    if i > 20:
        psf_frame += wfs.GetPSF(residual_opd)
        psf_frame_defocus_positive += wfs.GetPSF(residual_opd + defocus + astigmatism)
        psf_frame_defocus_negative += wfs.GetPSF(residual_opd - defocus + astigmatism)

        zernike_telemetry.append(coefs)


imshow(torch.stack((psf_frame, psf_frame_defocus_positive, psf_frame_defocus_negative)), figsize=(10,5))

In [ ]:
plt.semilogy(torch.stack(zernike_telemetry).squeeze().std(dim=0).cpu().detach())